In [1]:
%load_ext autoreload
%autoreload 2

# Automatic Fix AV classification
This notebook demonstrates how to fix a predicted AV classification map using topological recontrustion of the vascular tree.

In [ ]:
from pathlib import Path

import numpy as np
from jppype import Mosaic, vscode_theme

from fundus_vessels_toolkit import FundusData
from fundus_vessels_toolkit.pipelines import AVSegToTree, GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, fix_av_map, rasterize_tree_topology
from fundus_vessels_toolkit.utils.data_io import load_label_image
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

### Load a fundus image and its AV map

In [3]:
from fundus_vessels_toolkit.utils.data_io import load_image
from fundus_vessels_toolkit.vascular_data_objects.fundus_data import AVLabel


a = load_image("/home/gaby/Téléchargements/a.png", binarize=True)
v = load_image("/home/gaby/Téléchargements/v.png", binarize=True)
av = np.zeros(a.shape, dtype=np.uint8)
av[a] = AVLabel.ART
av[v] = AVLabel.VEI
av[a & v] = AVLabel.BOTH

fundus_test = FundusData(
    fundus="/home/gaby/Téléchargements/fundus.png",
    vessels=av,
    od="/home/gaby/Téléchargements/od.png",
    auto_resize=True,
)
# fundus_test = fundus_test.update(fundus=np.zeros_like(fundus_test.image))
fundus_test.draw()

[ WARN:0@0.145] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


View2D()

In [4]:
av2tree = GNNAVSegToTree()
trees = av2tree(fundus_test)

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/utils/bezier.py:404: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:204.)
  curve = torch.from_numpy(yx_points)


AttributeError: 'float' object has no attribute 'numpy'

In [ ]:
fundus_test2 = fundus_test.update(vessels=fix_av_map(fundus_test.av, trees))

m = Mosaic(2, cell_height=500)
fundus_test.draw(view=m[0])
draw_graph(av2tree.to_vgraph(fundus_test), m[0], av_attr="av")
fundus_test2.draw(view=m[1])
draw_trees(trees, m[1], edge_labels=False)
m

GridBox(children=(View2D(linkedTransformGroup='cf5cf59acfc9484b94cc840ff87f762b'), View2D(linkedTransformGroup…

In [5]:
m = Mosaic(1, cell_height=500)
fundus_test.draw(view=m[0])
draw_graph(av2tree.to_vgraph(fundus_test, simplify=False, label_av=True), m[0], av_attr="av")
m

NameError: name 'draw_graph' is not defined

In [ ]:
kwargs = dict(bridge_gap_smaller_than=40, fill_junctions=True)
a_map = rasterize_tree_topology(trees[0], **kwargs)[0] > 0
v_map = rasterize_tree_topology(trees[1], **kwargs)[0] > 0

m = Mosaic(2, cell_height=800, background=fundus_test.image)
m[0].add_label(a_map, name="a_map", color="red", opacity=0.5)
m[1].add_label(v_map, name="v_map", color="blue", opacity=0.5)
m

GridBox(children=(View2D(linkedTransformGroup='23ad4bdbb4d5458690975bd75a3a9da2'), View2D(linkedTransformGroup…

In [ ]:
STOP

NameError: name 'STOP' is not defined

In [ ]:
IMG = "g_007.png"
PATH = Path("/home/gaby/These/Data/Fundus/Vessels/GAVE/training/")


# Path to the raw fundus image
RAW_PATH = PATH / "images" / IMG

# Path to the artery/vein segmentation
AV_TRUE = PATH / "av" / IMG

AV_PRED = PATH / "Task1_2" / IMG

# Path to the OD segmentation
OD_PATH = PATH / "od" / IMG

fundus_gt = FundusData(fundus=RAW_PATH, vessels=AV_TRUE, od=OD_PATH)
fundus = fundus_gt.update(vessels=load_label_image(AV_PRED, ["black", "yellow", "cyan"]))

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=400)
fundus.draw(view=m[0])
fundus_gt.draw(view=m[1])
m

[ WARN:0@2.544] global loadsave.cpp:275 findDecoder imread_('/home/gaby/These/Data/Fundus/Vessels/GAVE/training/images/g_007.png'): can't open/read file: check file path/integrity


ValueError: Could not load image from /home/gaby/These/Data/Fundus/Vessels/GAVE/training/images/g_007.png

## Parse tree on the GT

Parse the topology of the ground truth segmentation and generate its topology map.

In [ ]:
seg2tree = NaiveAVSegToTree()
trees_gt = seg2tree(fundus_gt)

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

- The ``labels`` indicate to which subtree each vessel pixel belong, as well as the branching patterns to reach it.
- The ``topo`` map monotonically increases with the distance from the subtree root.

In [ ]:
m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=400,
)
fundus_gt.draw(view=m[0, 0])
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=True)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=True)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">VTree</h3>'), HTML(value=…

## Parse graph on the Prediction

In [ ]:
trees_fixed = AVSegToTree()(fundus)

fundus_fixed = fundus.update(vessels=fix_av_map(fundus.av, trees_fixed))

m = Mosaic(2, cols_titles=["Predicted", "Corrected"], cell_height=800)
fundus.draw(view=m[0])
fundus_fixed.draw(view=m[1])

draw_trees(trees_fixed, view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…